In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(33)

In [ ]:
inputs, answers = rd.DataSet.cubic().pair()
collector = rd.MetricCollector()

wait = (
    rd.Expr.when(rd.Expr.generation() > 10)
    .then(rd.Expr.select("scores.best").rolling(2).slope())
    .otherwise(0)
)

slope = rd.Expr.select("scores.best").rolling(5).slope() * 0.02

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.sum()],
        edge=rd.Op.weight(),
        output=rd.Op.sum(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .metrics(slope=slope)
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-11-14T18:25:04.763445Z  INFO State Change: PreStart -> Running
2026-11-14T18:25:04.763753Z  INFO Epoch 1    | Score:   2.9954 | Time: 158.62µs
2026-11-14T18:25:04.763957Z  INFO Epoch 2    | Score:   2.0038 | Time: 229.33µs
2026-11-14T18:25:04.764023Z  INFO Epoch 3    | Score:   2.0038 | Time: 279.12µs
2026-11-14T18:25:04.764082Z  INFO Epoch 4    | Score:   2.0038 | Time: 325.67µs
2026-11-14T18:25:04.764131Z  INFO Epoch 5    | Score:   2.0038 | Time: 365.12µs
2026-11-14T18:25:04.764199Z  INFO Epoch 6    | Score:   1.6821 | Time: 415.58µs
2026-11-14T18:25:04.764255Z  INFO Epoch 7    | Score:   1.6821 | Time: 462.92µs
2026-11-14T18:25:04.764309Z  INFO Epoch 8    | Score:   1.6821 | Time: 507.25µs
2026-11-14T18:25:04.764367Z  INFO Epoch 9    | Score:   1.6821 | Time: 556.75µs
2026-11-14T18:25:04.764433Z  INFO Epoch 10   | Score:   1.6821 | Time: 614.33µs
2026-11-14T18:25:04.764487Z  INFO Epoch 11   | Score:   1.6821 | Time: 657.16µs
2026-11-14T18:25:04.764537Z  INFO Epoch 12   | Score

In [11]:
collector.plot("scores.best", "pct.diversity")

In [4]:
eval_results = result.value.eval(inputs)
accuracy = rd.accuracy(result.value, inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.88%
	R² Score: 0.99994
	RMSE: 0.01610
	Loss (MSE): 0.00026
})

In [5]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",9.0,109.0,54.5,64.346718,4140.5,NaN,9.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000011,0.000064,0.000032,0.00003,9.0491e-10,NaN,0.000011,0.000053,2,64µs,32µs,30µs,10µs,53µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000006,0.000006,0.000006,0.0,0.0,NaN,0.000006,0.000006,1,6µs,6µs,0µs,6µs,6µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mutator.op.rate""",0.07,17.92,0.07,0.0,0.0,0.0,0.07,0.07,256,null,null,null,null,null,null,255,1,"[""alterer"", ""mutator"", … ""expr""]"
"""step.metric.time""",0.000008,0.00215,0.000008,0.000002,4.9495e-12,0.0,0.000007,0.000034,256,2150µs,8µs,2µs,7µs,34µs,0µs,255,1,"[""time"", ""step""]"
"""time""",0.00013,0.0203,0.000079,0.000032,1.0213e-9,0.0,0.000031,0.000215,256,20299µs,79µs,31µs,30µs,215µs,0µs,255,1,"[""time""]"


In [6]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
1
2
3
4
…
251
252
253


In [7]:
display(HTML(filtered._repr_html_()))

generation
i64
0
1
2
3
4
…
251
252
253
